# Gemini 3.0 Flash Evaluation for Trip Planning
This notebook evaluates the Gemini model on the `trip_planning_test.json` dataset using the `google-genai` SDK.

In [ ]:
import json
import time
import re
import os
from google import genai
from google.genai import types

# 1. Initialization
# Please set your API key in the environment or replace the string below.
API_KEY = os.environ.get("GEMINI_API_KEY", "YOUR_API_KEY_HERE")
client = genai.Client(api_key=API_KEY)

# File paths
DATA_PATH = "../data/trip_planning_test.json"
OUTPUT_DIR = "../data/gemini3-flash"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, "gemini_flash_evaluation_results.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)

system_instruction = (
    "You are a precise trip planning engine.\n"
    "TASK: Generate a valid travel itinerary that strictly satisfies all duration, sequence, and flight connectivity constraints.\n\n"
    "OUTPUT FORMAT (STRICT):\n"
    "- Use the format: '**Day X-Y:** Visit [City] for [N] days.' and '**Day X:** Fly from [City A] to [City B].'\n\n"
    "STRICT RULES:\n"
    "1. Output ONLY the final trip plan.\n"
    "2. NO explanations, NO reasoning traces, NO summaries, and NO conversational filler.\n"
    "3. DO NOT restate the task or constraints.\n"
    "4. Ensure the plan uses ONLY the direct flights listed.\n"
    "5. Keep the output concise and follow the example style EXACTLY.\n"
)

config = types.GenerateContentConfig(
    system_instruction=system_instruction,
    temperature=1.0,
)

In [ ]:
# 2. Helper Functions
def parse_itinerary_text(text):
    """
    Extracts the cities and start/end days from Gemini's output and formats them into a list of dictionaries.
    """
    parsed = []
    if not text:
        return parsed
        
    lines = text.split('\n')
    for line in lines:
        # Match day ranges: **Day 1-5:** or **Day 5:**
        match_day = re.search(r'\*\*Day\s+(\d+)(?:-(\d+))?:\*\*', line)
        if match_day:
            start = int(match_day.group(1))
            end = int(match_day.group(2)) if match_day.group(2) else start
            
            # Ignore flight lines
            if "Fly from" in line or "fly from" in line:
                continue
            
            # PERFECTED REGEX: Strictly captures the word(s) right before "for N days"
            # It ignores things like "Arriving in X and visit Y"
            match_city = re.search(r'(?:Visit|visit)\s+([a-zA-Z\s]+?)\s+for', line)
            
            # Fallback if it just said "in X for Y days"
            if not match_city:
                match_city = re.search(r'\s+in\s+([a-zA-Z\s]+?)\s+for', line)
                
            if match_city:
                city = match_city.group(1).strip().lower()
                parsed.append({"city": city, "start": start, "end": end})
    return parsed

def get_ground_truth(cities_str, durations_str):
    """
    Calculates the correct sequential start/end dates from the dataset strings.
    """
    cities = [c.strip().lower() for c in cities_str.split('**') if c.strip()]
    durations = [int(d) for d in durations_str.split('**') if d.strip()]
    
    ground_truth = []
    current_start = 1
    for city, duration in zip(cities, durations):
        current_end = current_start + duration - 1
        ground_truth.append({
            "city": city,
            "start": current_start,
            "end": current_end
        })
        current_start = current_end
    return ground_truth

In [ ]:
# 3. Evaluation Loop
MAX_RETRIES = 3
RETRY_WAIT_S = 30

def evaluate_dataset():
    with open(DATA_PATH, "r") as f:
        dataset = json.load(f)
    
    keys = list(dataset.keys())
    
    results = []
    total_correct = 0
    total_latency = 0.0
    total_in_tokens = 0
    total_out_tokens = 0
    
    # --- Check for existing progress ---
    if os.path.exists(OUTPUT_FILE):
        print(f"Found existing output file at {OUTPUT_FILE}. Resuming...")
        with open(OUTPUT_FILE, "r") as f:
            existing_data = json.load(f)
            all_logs = existing_data.get("detailed_logs", [])
            
            # Split logs: keep successful ones, discard failed/errored ones for retry
            successful_logs = [r for r in all_logs if r.get("error") is None]
            errored_logs = [r for r in all_logs if r.get("error") is not None]
            
            results = successful_logs
            
            if errored_logs:
                print(f"Found {len(errored_logs)} previously failed prompts - these will be retried!")
            
            # Re-calculate totals from SUCCESSFUL logs only
            total_correct = sum(1 for r in results if r.get("is_correct", False))
            total_latency = sum(r.get("latency", 0.0) for r in results)
            total_in_tokens = sum(r.get("input_tokens", 0) for r in results)
            total_out_tokens = sum(r.get("output_tokens", 0) for r in results)
            
    # Only skip prompts that have been processed successfully (no error)
    processed_keys = {r["id"] for r in results}
    if processed_keys:
        print(f"Skipping {len(processed_keys)} already-successful prompts.")
        print(f"Will process remaining {len(keys) - len(processed_keys)} prompts.\n")
    
    for i, key in enumerate(keys):
        if key in processed_keys:
            continue
            
        item = dataset[key]
        prompt = item["prompt_0shot"]
        cities_str = item["cities"]
        durations_str = item["durations"]
        
        ground_truth = get_ground_truth(cities_str, durations_str)
        
        raw_text = ""
        in_tokens = 0
        out_tokens = 0
        error_msg = None
        latency = 0.0
        
        # --- Retry Loop ---
        for attempt in range(1, MAX_RETRIES + 1):
            try:
                start_time = time.time()
                response = client.models.generate_content(
                    model='gemini-3-flash-preview',
                    contents=prompt,
                    config=config
                )
                latency = time.time() - start_time
                raw_text = response.text
                if response.usage_metadata:
                    in_tokens = response.usage_metadata.prompt_token_count
                    out_tokens = response.usage_metadata.candidates_token_count
                error_msg = None  # Success
                break  # Exit retry loop on success
            except Exception as e:
                latency = time.time() - start_time
                error_str = str(e)
                print(f"API Error on {key} (attempt {attempt}/{MAX_RETRIES}): {error_str}")
                error_msg = error_str
                if attempt < MAX_RETRIES:
                    print(f"Waiting {RETRY_WAIT_S}s before retry...")
                    time.sleep(RETRY_WAIT_S)
        
        parsed_output = parse_itinerary_text(raw_text)
        is_correct = (parsed_output == ground_truth) and (error_msg is None)
        
        if is_correct:
            total_correct += 1
            
        total_latency += latency
        total_in_tokens += in_tokens
        total_out_tokens += out_tokens
        
        # Record log
        results.append({
            "id": key,
            "prompt": prompt,
            "raw_text": raw_text,
            "parsed_output": parsed_output,
            "ground_truth": ground_truth,
            "is_correct": is_correct,
            "latency": latency,
            "input_tokens": in_tokens,
            "output_tokens": out_tokens,
            "error": error_msg
        })
        
        status = "ERROR" if error_msg else ("✓" if is_correct else "✗")
        print(f"Processed {i+1}/{len(keys)} [{status}] | Correct: {is_correct} | Latency: {latency:.2f}s")
        
        # --- Save Progress Immediately After Every Prompt ---
        num_samples = len(results)
        successful_samples = [r for r in results if r.get("error") is None]
        num_successful = len(successful_samples)
        summary = {
            "Total_Accuracy_%": (total_correct / num_samples * 100) if num_samples > 0 else 0,
            "Avg_Latency_s": total_latency / num_samples if num_samples > 0 else 0,
            "Avg_Input_Tokens": total_in_tokens / num_successful if num_successful > 0 else 0,
            "Avg_Output_Tokens": total_out_tokens / num_successful if num_successful > 0 else 0,
            "Total_Samples": num_samples,
            "Total_Errors": num_samples - num_successful
        }
        
        final_output = {
            "summary": summary,
            "detailed_logs": results
        }
        
        with open(OUTPUT_FILE, "w") as f:
            json.dump(final_output, f, indent=4)
            
        # Rate Limiting: Free Tier is 15 RPM, so sleep 4 seconds
        time.sleep(4)
        
    print("\n--- Evaluation Summary ---")
    print(json.dumps(summary, indent=4))
    print(f"Results completely saved to {OUTPUT_FILE}")

# Run the evaluation
evaluate_dataset()
